In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from collections import Counter

In [4]:
df_merged = pd.read_csv("/content/drive/MyDrive/CRIS/df_merged_full.csv")
brand_df = pd.read_csv("/content/drive/MyDrive/CRIS/brand_df.csv")
negative_df = pd.read_csv("/content/drive/MyDrive/CRIS/negative_brand_reviews.csv")

pivot = pd.read_csv("/content/drive/MyDrive/CRIS/brand_aspect_rating.csv", index_col=0)
neg_pivot = pd.read_csv("/content/drive/MyDrive/CRIS/brand_negative_rate.csv", index_col=0)
summary = pd.read_csv("/content/drive/MyDrive/CRIS/competitor_summary_table.csv", index_col=0)

business_recs = pd.read_csv("/content/drive/MyDrive/CRIS/business_recommendations.csv")
pm_summary_df = pd.read_csv("/content/drive/MyDrive/CRIS/pm_summary.csv")

In [5]:
print("df_merged:", df_merged.shape)
print("brand_df:", brand_df.shape)
print("negative_df:", negative_df.shape)
print("pivot:", pivot.shape)
print("neg_pivot:", neg_pivot.shape)
print("summary:", summary.shape)
print("business_recs:", business_recs.shape)
print("pm_summary_df:", pm_summary_df.shape)

df_merged: (12522, 32)
brand_df: (2901, 32)
negative_df: (478, 32)
pivot: (10, 8)
neg_pivot: (10, 8)
summary: (10, 5)
business_recs: (8, 7)
pm_summary_df: (10, 2)


In [6]:
documents = []
metadata = []

In [7]:
for brand, row in summary.iterrows():
    doc = f"""
Brand: {brand}
Total Reviews: {row['total_reviews']}
Average Review Rating: {row['avg_rating']}
Average Product Rating: {row['avg_product_rating']}
Average Price: {row['avg_price']}
Average Helpful Votes: {row['avg_helpful_vote']}
"""
    documents.append(doc.strip())
    metadata.append({
        "type": "brand_summary",
        "brand": brand
    })

In [8]:
for _, row in pm_summary_df.iterrows():
    documents.append(row["pm_summary"])
    metadata.append({
        "type": "pm_summary",
        "brand": row["brand"]
    })

In [9]:
for _, row in business_recs.iterrows():
    doc = f"""
Aspect: {row['aspect']}
Severity: {row['severity']}
Worst Brand: {row['worst_brand']}
Complaint Rate: {row['worst_negative_rate_pct']}%
Best Brand: {row['best_brand']}
Best Aspect Rating: {row['best_aspect_rating']}
Recommendation: {row['business_recommendation']}
"""
    documents.append(doc.strip())
    metadata.append({
        "type": "recommendation",
        "brand": row["worst_brand"],
        "aspect": row["aspect"]
    })

In [10]:
negative_reviews_unique = (
    negative_df.groupby(
        ["store", "parent_asin", "review_full", "rating", "importance_score"],
        as_index=False
    ).agg({
        "aspects": lambda x: sorted(set(x))
    })
)

negative_reviews_unique.head()
sample_size = min(2000, len(negative_reviews_unique))
negative_sample = negative_reviews_unique.sample(sample_size, random_state=42)

print("Unique negative review docs:", len(negative_sample))

Unique negative review docs: 160


In [11]:
for _, row in negative_sample.iterrows():
    doc = f"""
Brand: {row['store']}
Aspects: {", ".join(row['aspects'])}
Rating: {row['rating']}
Importance Score: {row['importance_score']:.2f}
Review: {row['review_full']}
"""
    documents.append(doc.strip())
    metadata.append({
        "type": "review_evidence",
        "brand": row["store"],
        "aspects": row["aspects"],
        "parent_asin": row["parent_asin"]
    })

In [12]:
positive_df = brand_df[brand_df["rating"] >= 4].copy()

positive_reviews_unique = (
    positive_df.groupby(
        ["store", "parent_asin", "review_full", "rating", "importance_score"],
        as_index=False
    ).agg({
        "aspects": lambda x: sorted(set(x))
    })
)

positive_reviews_unique.head()

,store,parent_asin,review_full,rating,importance_score,aspects
0,Anker,B019EZDT04,Great sound.. great customer service! Love Ank...,5.0,0.36,[sound]
1,Anker,B019EZDT04,High quality product. High quality company. I...,4.0,3.63,"[battery, build, connectivity]"
2,Anker,B019EZDT04,My boyfriend tried them and they stayed in fin...,5.0,0.54,"[comfort, sound]"
3,Anker,B01IUQERAY,Comfortable and sound good When you want to bl...,5.0,0.33,"[comfort, price, sound]"
4,Anker,B01NBOPJW2,Great value for bluetooth earbuds; fit like a ...,4.0,5.91,"[battery, build, comfort, connectivity, price,..."


In [13]:
positive_sample_size = min(1000, len(positive_reviews_unique))
positive_sample = positive_reviews_unique.sample(positive_sample_size, random_state=42)

print("Unique positive review docs:", len(positive_sample))

Unique positive review docs: 616


In [14]:
for _, row in positive_sample.iterrows():
    doc = f"""
Brand: {row['store']}
Aspects: {", ".join(row['aspects'])}
Rating: {row['rating']}
Importance Score: {row['importance_score']:.2f}
Review: {row['review_full']}
"""
    documents.append(doc.strip())
    metadata.append({
        "type": "positive_evidence",
        "brand": row["store"],
        "aspects": row["aspects"],
        "parent_asin": row["parent_asin"]
    })

In [15]:
print("Total documents:", len(documents))
print("Total metadata rows:", len(metadata))
print(Counter([m["type"] for m in metadata]))

Total documents: 804
Total metadata rows: 804
Counter({'positive_evidence': 616, 'review_evidence': 160, 'brand_summary': 10, 'pm_summary': 10, 'recommendation': 8})


In [16]:
seen_types = set()

for doc, meta in zip(documents, metadata):
    if meta["type"] not in seen_types:
        print("TYPE:", meta["type"])
        print("META:", meta)
        print(doc[:500])
        print("-" * 100)
        seen_types.add(meta["type"])

TYPE: brand_summary
META: {'type': 'brand_summary', 'brand': 'Anker'}
Brand: Anker
Total Reviews: 178.0
Average Review Rating: 4.230337078651686
Average Product Rating: 3.955056179775281
Average Price: nan
Average Helpful Votes: 0.9606741573033708
----------------------------------------------------------------------------------------------------
TYPE: pm_summary
META: {'type': 'pm_summary', 'brand': 'Anker'}
Anker has an average rating of 4.23 across 178 reviews. Its strongest feature is anc, while the most common customer complaint is sound. Improving sound could significantly enhance customer satisfaction.
----------------------------------------------------------------------------------------------------
TYPE: recommendation
META: {'type': 'recommendation', 'brand': 'Sony', 'aspect': 'anc'}
Aspect: anc
Severity: moderate
Worst Brand: Sony
Complaint Rate: 1.92%
Best Brand: Anker
Best Aspect Rating: 5.0
Recommendation: Anc is a moderate issue. Sony has the highest complaint rate at 1

In [17]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
doc_embeddings = embedding_model.encode(documents, show_progress_bar=True)
doc_embeddings = np.array(doc_embeddings).astype("float32")

print(doc_embeddings.shape)

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

(804, 384)


In [19]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print("Total indexed documents:", index.ntotal)

Total indexed documents: 804


In [20]:
def retrieve_documents(query, k=5):
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, k * 10)

    seen_reviews = set()
    seen_keys = set()
    results = []

    for rank_idx, idx in enumerate(indices[0]):
        meta = metadata[idx]
        doc = documents[idx]

        review_key = doc.split("Review:")[-1].strip() if "Review:" in doc else doc

        if "aspects" in meta:
            aspect_key = tuple(meta["aspects"])
        else:
            aspect_key = meta.get("aspect")

        key = (meta.get("brand"), aspect_key, meta.get("type"))

        if review_key in seen_reviews or key in seen_keys:
            continue

        seen_reviews.add(review_key)
        seen_keys.add(key)

        results.append({
            "rank": len(results) + 1,
            "document": doc,
            "metadata": meta,
            "distance": float(distances[0][rank_idx])
        })

        if len(results) >= k:
            break

    return results

In [21]:
query = "What are the biggest problems with Bose headphones?"
results = retrieve_documents(query, k=5)

for r in results:
    print("RANK:", r["rank"])
    print("META:", r["metadata"])
    print(r["document"])
    print("-" * 80)

RANK: 1
META: {'type': 'review_evidence', 'brand': 'Bose', 'aspects': ['battery', 'comfort', 'sound'], 'parent_asin': 'B0B62FJF1J'}
Brand: Bose
Aspects: battery, comfort, sound
Rating: 2.0
Importance Score: 9.93
Review: Nice sound and fit, but I got unlucky Maybe I was unlucky, but had several issues with this product: (1) headphone didn’t turn off, so battery life was pretty low; (2) volume button got stuck frequently and it would mute the headphone.<br /><br />I’ve tried force resetting a couple of times, didn’t solve the issue. Besides that, the product is amazing. I just think that when you get a $100 headset, it should be free of defects and I should not “fight” to make the product work, so I returned them
--------------------------------------------------------------------------------
RANK: 2
META: {'type': 'review_evidence', 'brand': 'Bose', 'aspects': ['battery', 'build', 'comfort', 'connectivity', 'microphone', 'price', 'sound'], 'parent_asin': 'B0B62FJF1J'}
Brand: Bose
Aspect

In [22]:
query = "Which brand is best for comfort?"
results = retrieve_documents(query, k=5)

for r in results:
    print("RANK:", r["rank"])
    print("META:", r["metadata"])
    print(r["document"])
    print("-" * 80)

RANK: 1
META: {'type': 'positive_evidence', 'brand': 'Sennheiser', 'aspects': ['comfort', 'sound'], 'parent_asin': 'B00WUN9PE2'}
Brand: Sennheiser
Aspects: comfort, sound
Rating: 5.0
Importance Score: 0.17
Review: Worth every penny I am 80 years old, these are very comfortable. sound is awesome....thank you
--------------------------------------------------------------------------------
RANK: 2
META: {'type': 'positive_evidence', 'brand': 'Sennheiser', 'aspects': ['comfort'], 'parent_asin': 'B00WUN9PE2'}
Brand: Sennheiser
Aspects: comfort
Rating: 5.0
Importance Score: 0.03
Review: Comfortable Very nice
--------------------------------------------------------------------------------
RANK: 3
META: {'type': 'positive_evidence', 'brand': 'Audio-Technica', 'aspects': ['comfort', 'sound'], 'parent_asin': 'B0BD99CPHB'}
Brand: Audio-Technica
Aspects: comfort, sound
Rating: 4.0
Importance Score: 3.07
Review: Buy this! Great sound and very comfortable.
-------------------------------------------

In [23]:
query = "Compare Sony and Bose on comfort and sound quality"
results = retrieve_documents(query, k=5)

for r in results:
    print("RANK:", r["rank"])
    print("META:", r["metadata"])
    print(r["document"])
    print("-" * 80)

RANK: 1
META: {'type': 'positive_evidence', 'brand': 'Sony', 'aspects': ['anc', 'build', 'comfort', 'connectivity', 'sound'], 'parent_asin': 'B088GJ1L5Z'}
Brand: Sony
Aspects: anc, build, comfort, connectivity, sound
Rating: 5.0
Importance Score: 1.02
Review: For Music? Get these Sony’s for sure. I have three pairs of these. They are very sturdy and heavy duty. My son is very rough on them and they take a licking and keep on ticking. I also have Bose QC35 II ..BUT these Sony’s definitely sound better than Bose. The noise cancellation is not as good as Bose IMO. If you like listening to music? Get these Sony’s. If noise canceling is more important than music quality then get Bose. Bose don’t sound near as good as these. I am a musician so sound quality is important.
--------------------------------------------------------------------------------
RANK: 2
META: {'type': 'positive_evidence', 'brand': 'Sony', 'aspects': ['comfort', 'sound'], 'parent_asin': 'B01MZFBKP7'}
Brand: Sony
Aspects:

In [28]:
def build_rag_prompt(query, k=5):
    retrieved = retrieve_documents(query, k=k)

    context = "\n\n".join(
        [f"[Document {i+1}]\n{item['document']}" for i, item in enumerate(retrieved)]
    )

    prompt = f"""
You are an AI based Review Intelligence System.

Use only the provided context to answer the user query.
Be specific, evidence-based, and concise.
If the answer is not supported by the context, say so clearly.

Context:
{context}

User Query:
{query}

Answer:
"""
    return prompt

In [29]:
prompt = build_rag_prompt("What are the biggest problems with Bose headphones?", k=5)
print(prompt)


You are an AI based Review Intelligence System.

Use only the provided context to answer the user query.
Be specific, evidence-based, and concise.
If the answer is not supported by the context, say so clearly.

Context:
[Document 1]
Brand: Bose
Aspects: battery, comfort, sound
Rating: 2.0
Importance Score: 9.93
Review: Nice sound and fit, but I got unlucky Maybe I was unlucky, but had several issues with this product: (1) headphone didn’t turn off, so battery life was pretty low; (2) volume button got stuck frequently and it would mute the headphone.<br /><br />I’ve tried force resetting a couple of times, didn’t solve the issue. Besides that, the product is amazing. I just think that when you get a $100 headset, it should be free of defects and I should not “fight” to make the product work, so I returned them

[Document 2]
Brand: Bose
Aspects: battery, build, comfort, connectivity, microphone, price, sound
Rating: 2.0
Importance Score: 10.78
Review: Return! I returned for the followi

In [30]:
def answer_with_retrieval_only(query, k=5):
    retrieved = retrieve_documents(query, k=k)

    answer = "Retrieved evidence:\n\n"
    for item in retrieved:
        answer += f"- {item['document']}\n\n"

    return answer

In [31]:
print(answer_with_retrieval_only("Why do users dislike Skullcandy?", k=5))

Retrieved evidence:

- Skullcandy has an average rating of 3.51 across 362 reviews. Its strongest feature is microphone, while the most common customer complaint is build. Improving build could significantly enhance customer satisfaction.

- Brand: Skullcandy
Total Reviews: 362.0
Average Review Rating: 3.505524861878453
Average Product Rating: 4.22707182320442
Average Price: 33.02408653846154
Average Helpful Votes: 1.5359116022099448

- Brand: Skullcandy
Aspects: price, sound
Rating: 5.0
Importance Score: 4.63
Review: Nice headphones These are very nice for your basic listening needs.  You cannot even tell you have them around your neck. The price was good too.  Now, if you are an audiophile that needs only the best there is...these are not for you.  They are perfect for the average person that wants to listen to some music.  Skullcandy never lets me down.

- Brand: Skullcandy
Aspects: comfort, sound
Rating: 1.0
Importance Score: 18.25
Review: Don't waste your time These headphones wer

In [32]:
corpus_df = pd.DataFrame({
    "document": documents,
    "metadata": [str(m) for m in metadata]
})

corpus_df.to_csv("/content/drive/MyDrive/CRIS/rag_corpus.csv", index=False)
np.save("/content/drive/MyDrive/CRIS/rag_embeddings.npy", doc_embeddings)
faiss.write_index(index, "/content/drive/MyDrive/CRIS/rag_index.faiss")

In [33]:
test_queries = [
    "What are the biggest problems with Bose headphones?",
    "Which brand is best for comfort?",
    "What should product teams fix first?",
    "Compare Sony and Audio-Technica on sound quality.",
    "Why do users dislike Skullcandy?"
]

test_results = []

for q in test_queries:
    retrieved = retrieve_documents(q, k=3)
    joined_docs = "\n\n".join([r["document"] for r in retrieved])

    test_results.append({
        "query": q,
        "retrieved_context": joined_docs
    })

test_df = pd.DataFrame(test_results)
test_df.to_csv("/content/drive/MyDrive/CRIS/rag_test_queries.csv", index=False)
test_df

,query,retrieved_context
0,What are the biggest problems with Bose headph...,"Brand: Bose\nAspects: battery, comfort, sound\..."
1,Which brand is best for comfort?,"Brand: Sennheiser\nAspects: comfort, sound\nRa..."
2,What should product teams fix first?,"Brand: Sony\nAspects: build, comfort, price\nR..."
3,Compare Sony and Audio-Technica on sound quality.,"Brand: Sony\nAspects: build, sound\nRating: 5...."
4,Why do users dislike Skullcandy?,Skullcandy has an average rating of 3.51 acros...


In [3]:
import os

required_files = [
    "/content/drive/MyDrive/CRIS/df_merged_full.csv",
    "/content/drive/MyDrive/CRIS/brand_aspect_rating.csv",
    "/content/drive/MyDrive/CRIS/brand_negative_rate.csv",
    "/content/drive/MyDrive/CRIS/competitor_summary_table.csv",
    "/content/drive/MyDrive/CRIS/business_recommendations.csv",
    "/content/drive/MyDrive/CRIS/pm_summary.csv",
    "/content/drive/MyDrive/CRIS/rag_corpus.csv",
    "/content/drive/MyDrive/CRIS/rag_embeddings.npy",
    "/content/drive/MyDrive/CRIS/rag_index.faiss",
    "/content/drive/MyDrive/CRIS/rag_test_queries.csv",
]

for f in required_files:
    print(os.path.exists(f), f)

True /content/drive/MyDrive/CRIS/df_merged_full.csv
True /content/drive/MyDrive/CRIS/brand_aspect_rating.csv
True /content/drive/MyDrive/CRIS/brand_negative_rate.csv
True /content/drive/MyDrive/CRIS/competitor_summary_table.csv
True /content/drive/MyDrive/CRIS/business_recommendations.csv
True /content/drive/MyDrive/CRIS/pm_summary.csv
True /content/drive/MyDrive/CRIS/rag_corpus.csv
True /content/drive/MyDrive/CRIS/rag_embeddings.npy
True /content/drive/MyDrive/CRIS/rag_index.faiss
True /content/drive/MyDrive/CRIS/rag_test_queries.csv
